# A typical ZipEats order: mean, SD, SE and interval

**Descriptive Statistics** · a worked example from the StatNex teaching guide

**The story.** ZipEats is a budget food-delivery app. Its operations team pulled 50 recent orders to get a feel for a "typical" delivery before a meeting with restaurant partners.

**The question.** What does a typical ZipEats order look like, and how precisely do 50 orders pin that down?

**To run it:**

1. Choose **Copy to Drive** at the top (or **File → Save a copy in Drive**). Your copy opens in a new tab. It is yours, so Colab runs it without a warning, and it keeps your changes.
2. In your copy, choose **Runtime → Run all**.

You can also run this page without a copy. Colab then warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `01_basic/descriptive_statistics.csv` (50 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
No.,Delivery time (min),Distance (km),Order total ($),Tip ($)
1.0,16.0,1.5,18.41,2.07
2.0,24.0,1.8,22.84,2.68
3.0,28.0,2.4,12.55,1.77
4.0,19.0,1.0,14.25,0.0
5.0,29.0,1.9,14.64,0.21
6.0,26.0,1.9,16.77,1.34
7.0,22.0,1.2,17.94,2.51
8.0,30.0,2.9,9.43,0.0
9.0,22.0,1.3,12.85,2.08
10.0,26.0,2.1,19.0,1.79
11.0,28.0,1.2,21.85,2.97
12.0,17.0,1.6,9.97,0.0
13.0,29.0,2.5,21.83,3.46
14.0,26.0,2.3,14.55,0.89
15.0,23.0,1.3,17.27,1.06
16.0,23.0,1.5,21.22,1.76
17.0,23.0,2.2,15.01,0.95
18.0,22.0,1.8,17.59,2.45
19.0,26.0,1.4,14.39,0.86
20.0,12.0,0.7,18.94,0.64
21.0,26.0,2.5,17.0,2.43
22.0,14.0,2.0,9.05,1.5
23.0,29.0,3.2,13.19,0.0
24.0,20.0,2.3,11.02,0.99
25.0,23.0,2.1,20.25,0.63
26.0,20.0,1.3,16.26,1.03
27.0,27.0,3.2,18.54,1.01
28.0,23.0,2.2,14.43,2.45
29.0,28.0,1.8,13.04,0.67
30.0,19.0,2.7,9.3,0.76
31.0,27.0,2.3,14.7,0.02
32.0,27.0,2.5,11.43,0.98
33.0,29.0,3.1,11.19,0.55
34.0,24.0,1.8,19.65,0.81
35.0,26.0,1.6,18.99,2.62
36.0,18.0,1.1,20.07,0.83
37.0,25.0,2.2,12.95,2.23
38.0,58.0,2.0,15.34,1.43
39.0,17.0,1.0,10.77,0.0
40.0,31.0,2.6,20.83,0.22
41.0,27.0,2.2,13.66,1.75
42.0,18.0,1.3,12.59,0.42
43.0,24.0,1.4,11.63,0.55
44.0,29.0,1.6,12.33,1.37
45.0,25.0,3.1,12.61,0.74
46.0,24.0,1.2,14.06,1.97
47.0,25.0,2.3,19.86,1.34
48.0,23.0,2.6,12.98,2.01
49.0,20.0,0.7,14.33,0.7
50.0,26.0,1.6,9.85,1.46

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)

## The data

In [ ]:
# Each column is read as numbers; blank and text cells are left out.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
columns = ["Delivery time (min)", "Distance (km)", "Order total ($)", "Tip ($)"]
series = {c: pd.to_numeric(data[c], errors="coerce").dropna().to_numpy(float) for c in columns}

# The data are a sample, so the variance divides by n - 1 (the unbiased variance).
ddof = 1

## Descriptive statistics

In [ ]:
# How to read the table: the mean is the centre; the standard deviation (sd) is how
# far a single value typically sits from the mean; the standard error (se = sd / √n,
# SciPy's stats.sem) is how far the mean itself would typically move if you drew
# another sample of the same size. The sd describes the values, the se the mean.
# Quartiles interpolate linearly between the sorted values (NumPy's default, Excel's
# QUARTILE.INC). Skewness and kurtosis are Excel's SKEW and KURT, the
# sample-corrected forms (bias=False), whichever composition is chosen.
# Geometric and harmonic means exist only for positive data, skewness and kurtosis
# only when the values vary; otherwise the cell is left empty (NaN).
def describe(x):
    n = x.size
    mean = np.mean(x)
    sd = np.std(x, ddof=ddof)
    q1, median, q3 = np.percentile(x, [25, 50, 75])
    positive = bool(np.all(x > 0))
    varies = x.max() > x.min()
    return {
        "n": n,
        "sum": np.sum(x),
        "mean": mean,
        "rms": np.sqrt(np.mean(x**2)),
        "geomean": stats.gmean(x) if positive else np.nan,
        "harmean": stats.hmean(x) if positive else np.nan,
        "sd": sd,
        "variance": np.var(x, ddof=ddof),
        "se": stats.sem(x, ddof=ddof),
        "range": x.max() - x.min(),
        "min": x.min(),
        "max": x.max(),
        "median": median,
        "q1": q1,
        "q3": q3,
        "iqr": q3 - q1,
        "cv": sd / mean if mean != 0 else np.nan,
        "kurtosis": stats.kurtosis(x, bias=False) if varies and n >= 4 else np.nan,
        "skewness": stats.skew(x, bias=False) if varies and n >= 3 else np.nan,
    }


summary = pd.DataFrame({name: describe(x) for name, x in series.items()})
# Only the statistics that were ticked, in the app's order.
chosen = ["n", "mean", "sd", "se"]
summary = summary.loc[[row for row in summary.index if row in chosen]]
print(summary)

## Confidence interval for the mean

In [ ]:
# mean ± t(n - 1) × SE, with t the two-sided critical value of Student's t on n - 1
# degrees of freedom; stats.t.interval does exactly this. The interval rests on the
# mean being roughly normal across samples: true for normal data, and close enough
# for most data once n is a few dozen, unless a few extreme values dominate. Read it
# as "the method catches the true mean in 95% of samples", not as a 95% chance for
# this one interval.
level = float(95) / 100
mean_ci = {}
for name, x in series.items():
    mean_ci[name] = list(stats.t.interval(level, x.size - 1, loc=np.mean(x), scale=stats.sem(x)))
mean_ci = pd.DataFrame(mean_ci, index=["lower", "upper"])
print(mean_ci)

## What the result says

The mean (the average) delivery time of the 50 orders is 24.46 minutes. The standard deviation, the typical distance of one order from the mean, is 6.49 minutes.

The standard error, 0.92 minutes, tells how far the sample mean may be from the true mean of all ZipEats orders. From it comes the 95% confidence interval for the mean: 22.6 to 26.3 minutes. This is the range of values the data agree with for the true mean.

So a typical order takes about 24 and a half minutes, and 50 orders pin that average down to within about 2 minutes. The same reading works for the other columns: the mean order total is \$15.26 and the mean tip is \$1.26.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.